# Colab reproduction notebook

This notebook reproduces and audits the calculations used in **Energy-weighted leakage limits work extraction in a symmetry-preserving quantum Otto engine**.

Edit `GITHUB_REPO` below after publishing the repository. The notebook clones the repository directly into Colab.

- **FAST audit (default):** checks the shipped CSV/JSON tables and regenerates the figures.
- **FULL recomputation:** reruns the spatial Schrödinger calculations, control sweeps, bias comparison, independent grid checks, validation, and figures.


In [ ]:
from pathlib import Path
import os, shutil, subprocess

GITHUB_REPO = 'https://github.com/YOUR_USERNAME/YOUR_REPOSITORY.git'
ROOT = Path('/content/quantum_otto_leakage')
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth','1',GITHUB_REPO,str(ROOT)], check=True)
os.chdir(ROOT)
print('Project root:', ROOT)


In [ ]:
!python -m pip install -q -r requirements.txt
import numpy, scipy, pandas, matplotlib, platform
print('Python', platform.python_version())
print('NumPy', numpy.__version__, 'SciPy', scipy.__version__, 'pandas', pandas.__version__, 'Matplotlib', matplotlib.__version__)


## Fast manuscript/data audit

This step is intentionally fast. It verifies the shipped numerical tables, thermodynamic checks, and the rounded values quoted in the manuscript.


In [ ]:
!python validate_results.py
!python paper_consistency_check.py


## Optional full recomputation

Set `FULL_RECOMPUTE=True` to regenerate the numerical study from the Hamiltonian rather than trusting the shipped tables. The full run can take several minutes or longer depending on the Colab CPU.


In [ ]:
FULL_RECOMPUTE = False

if FULL_RECOMPUTE:
    import subprocess, sys, time
    pipeline = [
        'run_study.py',
        'additional.py',
        'bias_tls_compare.py',
        'verify_independent.py',
        'validate_results.py',
        'paper_consistency_check.py',
        'make_figures.py',
        'make_bias_figure.py',
    ]
    t0 = time.time()
    for script in pipeline:
        print('\n===', script, '===')
        subprocess.run([sys.executable, script], check=True)
    print(f'Full pipeline finished in {(time.time()-t0)/60:.1f} min')
else:
    print('FULL_RECOMPUTE=False: using the verified shipped numerical tables.')


## Regenerate figures from the current numerical tables


In [ ]:
!python make_figures.py
!python make_bias_figure.py
print('Figures regenerated in', ROOT/'figures')


## Compile the LaTeX manuscript with the bibliography

The build sequence explicitly runs BibTeX so that all citations and the reference list appear in the PDF.


In [ ]:
import shutil, subprocess, os
if shutil.which('pdflatex') is None or shutil.which('bibtex') is None:
    !apt-get -qq update
    !apt-get -qq install -y texlive-latex-base texlive-latex-extra texlive-fonts-recommended texlive-science texlive-bibtex-extra >/dev/null

cmds = [
    ['pdflatex','-interaction=nonstopmode','manuscript.tex'],
    ['bibtex','manuscript'],
    ['pdflatex','-interaction=nonstopmode','manuscript.tex'],
    ['pdflatex','-interaction=nonstopmode','manuscript.tex'],
]
for cmd in cmds:
    r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if r.returncode:
        print(r.stdout[-5000:])
        raise RuntimeError('Build failed: ' + ' '.join(cmd))
log = Path('manuscript.log').read_text(errors='ignore')
for bad in ['undefined citations', 'There were undefined references', 'Citation `']:
    if bad in log:
        raise RuntimeError('LaTeX citation/reference warning remains: ' + bad)
print('PDF compiled:', ROOT/'manuscript.pdf')


In [ ]:
from IPython.display import IFrame, display
display(IFrame(str(ROOT/'manuscript.pdf'), width='100%', height=700))


## Download the reproduced PDF


In [ ]:
from google.colab import files
files.download(str(ROOT/'manuscript.pdf'))
